# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata (access .name and .description attributes)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s as provided in the schema.

In [ ]:
# List all record sets, fields, columns, and their @id
print("Record Sets and their Fields/Columns IDs:")

for rset in dataset.record_sets():
    print(f"- Record Set Name: {rset.name} (@id: {rset.id})")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - Field Name: {field.name} (@id: {field.id}) [data_type: {field.data_type}]")
        if hasattr(field, 'column') and field.column is not None:
            print(f"      Column: {field.column.name} (@id: {field.column.id})")
    print()
# For illustration, print the first 2 records for each record set

for rset in dataset.record_sets():
    print(f"Sample records from: {rset.name} (@id: {rset.id})")
    for i, record in enumerate(dataset.records(record_set=rset.id)):
        print(record)
        if i == 1:
            break
    print("-")

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis, using each record set's `@id`.

In [ ]:
# Extract all record set IDs for programmatic access
record_set_ids = [rset.id for rset in dataset.record_sets()]
print("Record set @ids found in dataset:", record_set_ids)

# Load each record set into a DataFrame, keyed by its @id
dataframes = dict()
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    if not df.empty:
        print(f"Columns for record set '{rset_id}':", df.columns.tolist())

# For this example, pick the first non-empty record set for EDA below:
main_record_set = None
for rset_id in record_set_ids:
    if not dataframes[rset_id].empty:
        main_record_set = rset_id
        break
if main_record_set is None:
    raise RuntimeError("No non-empty record set found!")
print(f"Main record set for further analysis: {main_record_set}")
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering, normalization, grouping, handling outliers, etc. Reference field and record set by their `@id`.

In [ ]:
# Pick a numeric field automatically if possible

df = dataframes[main_record_set]
# List numeric-like columns
numeric_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()
if len(numeric_cols) == 0:
    raise RuntimeError("No numeric columns found for EDA.")

numeric_field_id = numeric_cols[0]
print(f"Using numeric field (by column name, also @id): {numeric_field_id}")

# Example: Filter for values > threshold (tailor as appropriate)
threshold = df[numeric_field_id].median()  # Use median as a reasonable threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records in '{main_record_set}' with '{numeric_field_id}' > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a suitable categorical column (not the index)
group_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field_id = None
for col in group_cols:
    n_unique = df[col].nunique()
    if 2 <= n_unique <= 10:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped filtered results by '{group_field_id}' (also field @id):")
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize the distribution of a numeric field and the grouped means by a categorical field, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} per {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore a Croissant-conformant mental health dataset from Kilifi County, Kenya, using the `mlcroissant` library. We loaded dataset metadata, inspected record sets and fields using their `@id`, extracted a main record set, performed basic EDA including filtering and normalization, and visualized results. The dataset includes a rich set of field and record set identifiers that make programmatic, standards-based data science workflows possible.